# Module 3: Advanced Relational Analytical Engine & Superstore Mining
**Author:** Rakshit Gupta  
**Objective:** Build enterprise normalization structures, evaluate multi-level subqueries, initialize Common Table Expressions (CTEs), and run analytical Window Functions mapping.

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import os

print("[System Log] Initializing standalone memory database channel...")
session_db = sqlite3.connect('superstore_analytics.db')
engine = session_db.cursor()

engine.executescript("""
DROP TABLE IF EXISTS orders;
DROP TABLE IF EXISTS products;
DROP TABLE IF EXISTS customers;
DROP TABLE IF EXISTS superstore_raw;
""")

engine.execute("""
CREATE TABLE superstore_raw (
    Row_ID INT PRIMARY KEY, Order_ID VARCHAR(20), Order_Date DATE, Ship_Date DATE,
    Ship_Mode VARCHAR(50), Customer_ID VARCHAR(20), Customer_Name VARCHAR(100),
    Segment VARCHAR(50), Country VARCHAR(50), City VARCHAR(50), State VARCHAR(50),
    Postal_Code VARCHAR(20), Region VARCHAR(50), Product_ID VARCHAR(20),
    Category VARCHAR(50), Sub_Category VARCHAR(50), Product_Name VARCHAR(200),
    Sales DECIMAL(10,4), Quantity INT, Discount DECIMAL(5,2), Profit DECIMAL(10,4)
);
""")

csv_path = 'archive (1).zip/Sample - Superstore.csv'
alt_path = 'Sample - Superstore.csv'

if os.path.exists(csv_path):
    print(f"[Pipeline Log] Discovered raw source artifact at path: '{csv_path}'")
    staging_df = pd.read_csv(csv_path, encoding='windows-1252')
    staging_df.columns = staging_df.columns.str.replace(' ', '_').str.replace('-', '_')
    staging_df.to_sql('superstore_raw', session_db, if_exists='append', index=False)
    print(f"[Pipeline Log] Success! Ingested {len(staging_df)} production records.")
elif os.path.exists(alt_path):
    print(f"[Pipeline Log] Discovered raw source artifact at alternative path: '{alt_path}'")
    staging_df = pd.read_csv(alt_path, encoding='windows-1252')
    staging_df.columns = staging_df.columns.str.replace(' ', '_').str.replace('-', '_')
    staging_df.to_sql('superstore_raw', session_db, if_exists='append', index=False)
    print(f"[Pipeline Log] Success! Ingested {len(staging_df)} production records via fallback.")
else:
    print("[Warning] Ingestion path block empty. Hydrating baseline evaluation mock vectors...")
    mock_records = [
        (1, 'CA-2016-152156', '2016-11-08', '2016-11-11', 'Second Class', 'CG-12520', 'Claire Gute', 'Consumer', 'United States', 'Henderson', 'Kentucky', '42420', 'South', 'FUR-CH-10002024', 'Furniture', 'Chairs', 'Hon Deluxe Chair', 731.94, 3, 0.0, 219.58),
        (2, 'CA-2016-138688', '2016-06-12', '2016-06-16', 'Second Class', 'DV-13045', 'Darrin Van Huff', 'Corporate', 'United States', 'Los Angeles', 'California', '90036', 'West', 'OFF-BI-10003527', 'Office Supplies', 'Labels', 'Self-Adhesive Labels', 957.57, 2, 0.0, 430.22),
        (3, 'CA-2015-115812', '2015-06-09', '2015-06-14', 'Standard Class', 'BH-11710', 'Brosina Hoffman', 'Consumer', 'United States', 'Los Angeles', 'California', '90036', 'West', 'TEC-CO-10004722', 'Technology', 'Phones', 'iPhone 13 Pro Max', 1706.18, 4, 0.2, 680.12),
        (4, 'CA-2014-143336', '2014-08-27', '2014-09-01', 'Second Class', 'ZS-21895', 'Zuschlich Donatelli', 'Consumer', 'United States', 'San Francisco', 'California', '94109', 'West', 'OFF-AR-10002833', 'Office Supplies', 'Art', 'Premium Pencils', 507.72, 3, 0.0, 120.40),
        (5, 'CA-2016-117590', '2016-12-08', '2016-12-11', 'First Class', 'GH-14485', 'Gene Hale', 'Corporate', 'United States', 'Richardson', 'Texas', '75080', 'Central', 'TEC-MA-10002412', 'Technology', 'Copiers', 'Canon Advance Copier', 22638.48, 5, 0.2, 8100.50),
        (6, 'CA-2016-120933', '2016-04-12', '2016-04-17', 'Standard Class', 'SM-20320', 'Sean Miller', 'Home Office', 'United States', 'Jacksonville', 'Florida', '32216', 'South', 'TEC-MA-10004125', 'Technology', 'Machines', '3D Production Engine', 17499.95, 2, 0.5, -3200.00),
        (7, 'CA-2015-144933', '2015-10-18', '2015-10-22', 'First Class', 'TC-21295', 'Tamara Chand', 'Corporate', 'United States', 'Lafayette', 'Indiana', '47905', 'Central', 'OFF-BI-10002444', 'Office Supplies', 'Binders', 'Heavy Duty Binder Rings', 9099.93, 7, 0.0, 4100.00)
    ]
    engine.executemany("INSERT INTO superstore_raw VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)", mock_records)

engine.executescript("""
CREATE TABLE customers AS SELECT DISTINCT Customer_ID, Customer_Name, Segment FROM superstore_raw;
CREATE TABLE products AS SELECT DISTINCT Product_ID, Product_Name, Category, Sub_Category FROM superstore_raw;
CREATE TABLE orders AS SELECT Row_ID, Order_ID, Customer_ID, Product_ID, Order_Date, Sales, Quantity, Profit FROM superstore_raw;
""")
session_db.commit()
print("[Pipeline Log] 3NF Relational Tables (customers, products, orders) generated cleanly.")

In [ ]:
print("\n==================================================================")
print("       STEP 2: SUBQUERIES AND CTEs EXECUTION LAYER                ")
print("==================================================================")

print("--- TASK 2.1: Transactions Exceeding Global Sales Average ---")
q2_1 = "SELECT order_id, customer_id, sales FROM orders WHERE sales > (SELECT AVG(sales) FROM orders) LIMIT 5;"
display(pd.read_sql_query(q2_1, session_db))

print("\n--- TASK 2.2: Highest Sales Transaction Instance Per Customer ---")
q2_2 = """
SELECT o1.customer_id, o1.order_id, o1.sales 
FROM orders o1 
WHERE o1.sales = (SELECT MAX(o2.sales) FROM orders o2 WHERE o2.customer_id = o1.customer_id)
LIMIT 5;
"""
display(pd.read_sql_query(q2_2, session_db))

print("\n--- TASK 2.3: Total Sales For Each Customer (CTE) ---")
q2_3 = """
WITH CustomerSales_CTE AS (
    SELECT customer_id, SUM(sales) as total_sales FROM orders GROUP BY customer_id
)
SELECT c.customer_name, ROUND(cte.total_sales, 2) as aggregated_sales 
FROM CustomerSales_CTE cte
INNER JOIN customers c ON cte.customer_id = c.customer_id
LIMIT 5;
"""
display(pd.read_sql_query(q2_3, session_db))

print("\n--- TASK 2.4: Customers Whose Total Sales Are Above Average (CTE + Subquery) ---")
q2_4 = """
WITH CustomerTotal_CTE AS (
    SELECT customer_id, SUM(sales) as total_sales FROM orders GROUP BY customer_id
)
SELECT c.customer_name, ROUND(cte.total_sales, 2) as profile_sales 
FROM CustomerTotal_CTE cte
INNER JOIN customers c ON cte.customer_id = c.customer_id
WHERE cte.total_sales > (
    SELECT AVG(total_sales) FROM (SELECT SUM(sales) as total_sales FROM orders GROUP BY customer_id)
)
LIMIT 5;
"""
display(pd.read_sql_query(q2_4, session_db))

In [ ]:
print("\n==================================================================")
print("       STEP 2 CONTINUED & STEP 3: ADVANCED WINDOW FUNCTIONS      ")
print("==================================================================")

print("--- TASK 2.5: Rank All Customers Based On Total Sales ---")
q2_5 = """
WITH CustomerRank_CTE AS (
    SELECT customer_id, SUM(sales) as total_sales FROM orders GROUP BY customer_id
)
SELECT c.customer_name, ROUND(cte.total_sales, 2) as total_sales,
       DENSE_RANK() OVER (ORDER BY cte.total_sales DESC) as customer_rank
FROM CustomerRank_CTE cte
INNER JOIN customers c ON cte.customer_id = c.customer_id
LIMIT 5;
"""
display(pd.read_sql_query(q2_5, session_db))

print("\n--- TASK 2.6: Assign Row Numbers To Each Order Within A Customer ---")
q2_6 = """
SELECT order_id, customer_id, sales,
       ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY sales DESC) as row_num_seq
FROM orders LIMIT 5;
"""
display(pd.read_sql_query(q2_6, session_db))

print("\n--- TASK 2.7 & STEP 3: Display Top Customers & Final Combined Query Mapping ---")
q3_combined = """
WITH FinalSummary_CTE AS (
    SELECT customer_id, SUM(sales) AS total_sales
    FROM orders
    GROUP BY customer_id
)
SELECT c.customer_name,
       ROUND(cte.total_sales, 2) AS total_sales,
       DENSE_RANK() OVER (ORDER BY cte.total_sales DESC) AS customer_rank
FROM FinalSummary_CTE cte
INNER JOIN customers c ON cte.customer_id = c.customer_id
ORDER BY customer_rank ASC LIMIT 5;
"""
display(pd.read_sql_query(q3_combined, session_db))

In [ ]:
print("\n==================================================================")
print("       MINI PROJECT: CUSTOMER SALES INSIGHTS REPORT               ")
print("==================================================================")

print("--- 1. Who are the top 5 customers? ---")
top_5 = """
WITH Agg AS (SELECT customer_id, SUM(sales) as ts FROM orders GROUP BY customer_id)
SELECT c.customer_name, ROUND(a.ts, 2) as total_sales FROM Agg a 
INNER JOIN customers c ON a.customer_id = c.customer_id ORDER BY ts DESC LIMIT 5;
"""
display(pd.read_sql_query(top_5, session_db))

print("\n--- 2. Who are the bottom 5 customers? ---")
bottom_5 = """
WITH Agg AS (SELECT customer_id, SUM(sales) as ts FROM orders GROUP BY customer_id)
SELECT c.customer_name, ROUND(a.ts, 2) as total_sales FROM Agg a 
INNER JOIN customers c ON a.customer_id = c.customer_id ORDER BY ts ASC LIMIT 5;
"""
display(pd.read_sql_query(bottom_5, session_db))

print("\n--- 3. Which customers made only one order? ---")
single_order = """
SELECT c.customer_name, COUNT(DISTINCT o.order_id) as order_count FROM orders o
INNER JOIN customers c ON o.customer_id = c.customer_id
GROUP BY o.customer_id HAVING order_count = 1 LIMIT 5;
"""
display(pd.read_sql_query(single_order, session_db))

print("\n--- 4. Which customers have above-average sales? ---")
above_avg = """
SELECT DISTINCT c.customer_name FROM orders o
INNER JOIN customers c ON o.customer_id = c.customer_id
WHERE o.sales > (SELECT AVG(sales) FROM orders) LIMIT 5;
"""
display(pd.read_sql_query(above_avg, session_db))

print("\n--- 5. What is the highest order value per customer? ---")
max_order = """
SELECT c.customer_name, ROUND(MAX(o.sales), 2) as max_single_order_value FROM orders o
INNER JOIN customers c ON o.customer_id = c.customer_id
GROUP BY o.customer_id LIMIT 5;
"""
display(pd.read_sql_query(max_order, session_db))
session_db.close()